# Re:Desert — Model A · Random Forest surface classification
This notebook trains **only after** human-reviewed labels have been exported from the annotation tool. Predictions are land-cover classes—not native plant survival, soil types, or restoration approval.

**Required files:** `lahbab_hls_features_corrected.tif`, `redesert_modelA_verified_labels.csv`. Do not use unlabeled candidates as ground truth.

In [ ]:
# In Colab: upload the two files listed above.
from google.colab import files
uploaded = files.upload()

Saving lahbab_hls_features_corrected.tif to lahbab_hls_features_corrected (1).tif
Saving redesert_modelA_verified_labels.csv to redesert_modelA_verified_labels.csv


In [ ]:
%pip -q install rasterio scikit-learn pandas joblib pillow

In [ ]:
"""Re:Desert Model A — supervised Random Forest on independently reviewed geospatial labels.
Run: python train_model_a.py --raster lahbab_hls_features_corrected.tif --labels redesert_modelA_verified_labels.csv
Labels must be human reviewed; this script deliberately does not invent ground truth.
"""
from __future__ import annotations
import argparse, json
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import rasterio
from PIL import Image
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import confusion_matrix, classification_report, f1_score, balanced_accuracy_score
from sklearn.model_selection import GroupShuffleSplit

BANDS = ['B2','B3','B4','B8A','B11','B12','NDVI','MSAVI']
CLASSES = {0:'Bare/sparse', 1:'Vegetated', 2:'Built/disturbed'}
COLORS = np.array([[217,180,117],[52,136,76],[98,112,126]], dtype=np.uint8)

def train(raster_path: str, label_csv: str, output_dir: str = 'model_A_results'):
    dst=Path(output_dir); dst.mkdir(parents=True,exist_ok=True)
    labels=pd.read_csv(label_csv)
    req={'row','col','label'}
    if not req.issubset(labels.columns):
        raise ValueError('CSV needs columns '+str(sorted(req))+'. Export from the annotation tool.')
    labels=labels.dropna(subset=['row','col','label']).copy()
    for c in ['row','col','label']: labels[c]=pd.to_numeric(labels[c],errors='raise').astype(int)
    labels=labels.drop_duplicates(subset=['row','col'],keep='last')
    if not labels['label'].isin(CLASSES).all(): raise ValueError('Labels must be exactly 0, 1 or 2.')
    counts=labels['label'].value_counts().to_dict()
    print('Verified samples:',len(labels),'by class:',{CLASSES[k]:counts.get(k,0) for k in CLASSES})
    if min(counts.get(c,0) for c in CLASSES)<10:
        raise ValueError('Need at least 10 independently reviewed samples PER class even for a smoke test. Prefer >=30–50 per class for an MVP, across separate locations.')
    with rasterio.open(raster_path) as ds:
        if ds.count!=8: raise ValueError('Expected 8 bands.')
        data=ds.read().astype(np.float32)
        if not ((labels.row>=0)&(labels.row<ds.height)&(labels.col>=0)&(labels.col<ds.width)).all():
            raise ValueError('Some point row/col indices are outside the GeoTIFF. Use matching Re:Desert label tool and TIFF.')
        feats=data[:, labels['row'].to_numpy(), labels['col'].to_numpy()].T
        valid=np.isfinite(feats).all(axis=1)
        if not valid.all():print('Dropping missing reflectance samples', (~valid).sum())
        labels=labels.iloc[np.where(valid)[0]].copy()
        feats=feats[valid]
        all_features=data.reshape(ds.count,-1).T
        h,w=ds.height,ds.width
        profile=ds.profile.copy()
    y=labels.label.to_numpy()
    # Geographic blocks prevent nearby pixels appearing in both train & test.
    # Block width 35 pixels ~ 1 km; use one split with all classes in both portions.
    block_width=35
    groups=(labels.row.to_numpy()//block_width)*100+(labels.col.to_numpy()//block_width)
    splits=GroupShuffleSplit(n_splits=200,test_size=0.25,random_state=1729)
    split=None
    for tr,te in splits.split(feats,y,groups):
        if set(y[tr])==set(CLASSES) and set(y[te])==set(CLASSES) and len(np.unique(groups[te]))>=2:
            split=(tr,te);break
    if split is None:
        raise ValueError('Cannot create a geographically separated train/test split containing all classes. Label more locations spread across the study area; do not switch to a random pixel split.')
    tr,te=split
    model=RandomForestClassifier(n_estimators=150,min_samples_leaf=2,max_features='sqrt',class_weight='balanced_subsample',n_jobs=-1,random_state=21)
    baseline=DummyClassifier(strategy='most_frequent')
    model.fit(feats[tr],y[tr]);baseline.fit(feats[tr],y[tr])
    pred=model.predict(feats[te]);base_pred=baseline.predict(feats[te])
    report=classification_report(y[te],pred,labels=[0,1,2],target_names=[CLASSES[i] for i in CLASSES],zero_division=0,output_dict=True)
    print(classification_report(y[te],pred,labels=[0,1,2],target_names=[CLASSES[i] for i in CLASSES],zero_division=0))
    scores={
      'verified_training_size':int(len(tr)),'verified_test_size':int(len(te)),
      'unique_training_blocks':int(len(np.unique(groups[tr]))),'unique_test_blocks':int(len(np.unique(groups[te]))),
      'class_counts_all':{CLASSES[c]:int((y==c).sum()) for c in CLASSES},
      'class_counts_test':{CLASSES[c]:int((y[te]==c).sum()) for c in CLASSES},
      'macro_f1':float(f1_score(y[te],pred,average='macro',zero_division=0)),
      'balanced_accuracy':float(balanced_accuracy_score(y[te],pred)),
      'baseline_macro_f1':float(f1_score(y[te],base_pred,average='macro',zero_division=0)),
      'classification_report':report,
      'important_disclaimer':'Labels human reviewed; surface class is not a measure of native status, vegetation persistence, soil texture or planting survival.'
    }
    (dst/'model_A_metrics.json').write_text(json.dumps(scores,indent=2))
    joblib.dump({'model':model,'bands':BANDS,'classes':CLASSES},dst/'model_A_random_forest.joblib')
    validpix=np.isfinite(all_features).all(axis=1)
    class_img=np.full(h*w,255,dtype=np.uint8)
    class_img[validpix]=model.predict(all_features[validpix]).astype('uint8')
    class_img=class_img.reshape(h,w)
    with rasterio.open(dst/'model_A_classification.tif','w',driver='GTiff',height=h,width=w,count=1,dtype='uint8',crs=profile['crs'],transform=profile['transform'],nodata=255,compress='deflate') as out:
        out.write(class_img,1);out.set_band_description(1,'predicted_surface_class')
    color_img=np.full((h,w,3),240,dtype='uint8')
    for k in CLASSES:color_img[class_img==k]=COLORS[k]
    Image.fromarray(color_img).resize((834,706),Image.Resampling.NEAREST).save(dst/'model_A_classification_preview.png')
    np.savetxt(dst/'model_A_confusion_matrix.csv',confusion_matrix(y[te],pred,labels=[0,1,2]),fmt='%d',delimiter=',',header='pred_bare,pred_vegetated,pred_disturbed',comments='')
    print('\nTest macro F1:',round(scores['macro_f1'],3),'Baseline F1:',round(scores['baseline_macro_f1'],3))
    print('Saved results in',dst.absolute())
    return scores



In [ ]:
metrics = train('lahbab_hls_features_corrected.tif', 'redesert_modelA_verified_labels.csv', 'model_A_results')
print('Macro F1:', round(metrics['macro_f1'], 3))

Verified samples: 290 by class: {'Bare/sparse': 171, 'Vegetated': 94, 'Built/disturbed': 25}
                 precision    recall  f1-score   support

    Bare/sparse       0.79      0.67      0.72        45
      Vegetated       0.49      0.73      0.59        30
Built/disturbed       1.00      0.62      0.76        21

       accuracy                           0.68        96
      macro avg       0.76      0.67      0.69        96
   weighted avg       0.74      0.68      0.69        96


Test macro F1: 0.691 Baseline F1: 0.213
Saved results in /content/model_A_results
Macro F1: 0.691


In [ ]:
from google.colab import files
import shutil
shutil.make_archive('ReDesert_ModelA_Results', 'zip', 'model_A_results')
files.download('ReDesert_ModelA_Results.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>